### ETL Final

In [21]:
import os
import time
import psycopg2

# ─────────────────────────────────────────────────────────────────
# Conexión
# ─────────────────────────────────────────────────────────────────
HOST     = os.environ.get("DB_HOST", "seal-users.c180so2u4aci.us-east-2.rds.amazonaws.com")
PORT     = os.environ.get("DB_PORT", "5432")
USER     = os.environ.get("DB_USER", "postgres")
PASSWORD = os.environ.get("DB_PASSWORD", "Nomelase123+")


def uri(dbname):
    return f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{dbname}"


def dblink_conn(dbname):
    return f"host={HOST} port={PORT} dbname={dbname} user={USER} password={PASSWORD}"


URI_USUARIOS      = uri("UsersETL")
URI_CANCIONES     = uri("CancionETL")
URI_INTERACCIONES = uri("Interacciones")
URI_OLAP          = uri("OLAP_Operations")


def ejecutar(uri_, sql, msg=""):
    """Ejecuta SQL directo en la BD destino y mide el tiempo."""
    t = time.time()
    conn = psycopg2.connect(uri_)
    conn.autocommit = True
    cur = conn.cursor()
    try:
        cur.execute(sql)
        rows = cur.rowcount
    finally:
        cur.close()
        conn.close()
    print(f"   [OK] {msg} → {rows if rows > 0 else '?'} filas en {time.time()-t:.1f}s")
    return rows


def ejecutar_fetch(uri_, sql):
    conn = psycopg2.connect(uri_)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return rows


# ─────────────────────────────────────────────────────────────────
# PASO 0: dblink
# ─────────────────────────────────────────────────────────────────
def instalar_dblink():
    print("\n[0] Instalando extensión dblink en OLAP_Operations...")
    ejecutar(URI_OLAP, "CREATE EXTENSION IF NOT EXISTS dblink;", "dblink instalado")


# ─────────────────────────────────────────────────────────────────
# PASO 1: Limpiar OLAP
# ─────────────────────────────────────────────────────────────────
def limpiar_olap():
    print("\n[1] Limpiando tablas OLAP...")
    sql = """
    DO $$ BEGIN
        TRUNCATE TABLE hechos_simulaciones   RESTART IDENTITY CASCADE;
        TRUNCATE TABLE hechos_data_quality   RESTART IDENTITY CASCADE;
        TRUNCATE TABLE hechos_interacciones  RESTART IDENTITY CASCADE;
        TRUNCATE TABLE hechos_adquisicion    RESTART IDENTITY CASCADE;
        TRUNCATE TABLE dim_cancion           RESTART IDENTITY CASCADE;
        TRUNCATE TABLE dim_dispositivo       RESTART IDENTITY CASCADE;
        TRUNCATE TABLE dim_fecha_hora        RESTART IDENTITY CASCADE;
        TRUNCATE TABLE dim_usuario           RESTART IDENTITY CASCADE;
        TRUNCATE TABLE dim_ubicacion         RESTART IDENTITY CASCADE;
    END $$;
    """
    ejecutar(URI_OLAP, sql, "OLAP limpio")


# ─────────────────────────────────────────────────────────────────
# PASO 2: dim_escenario
# ─────────────────────────────────────────────────────────────────
def cargar_dim_escenario():
    print("\n[2] Cargando dim_escenario (catálogo de escenarios DSS)...")
    sql = """
        INSERT INTO dim_escenario (nombre_escenario, descripcion)
        SELECT v.nombre, v.descripcion
        FROM (VALUES
            ('Base',      'Proyección manteniendo la tendencia histórica observada'),
            ('Optimista', 'Proyección asumiendo un incremento de actividad del 15%% mensual'),
            ('Pesimista', 'Proyección asumiendo una caída de actividad del 10%% mensual')
        ) AS v(nombre, descripcion)
        WHERE NOT EXISTS (
            SELECT 1 FROM dim_escenario de WHERE de.nombre_escenario = v.nombre
        );
    """
    ejecutar(URI_OLAP, sql, "dim_escenario")


# ─────────────────────────────────────────────────────────────────
# PASO 3: Dimensiones de forma incremental (Sin Duplicados)
# ─────────────────────────────────────────────────────────────────
def cargar_dimensiones():
    print("\n[3] Cargando dimensiones via dblink de forma incremental...")

    conn_usr = dblink_conn("UsersETL")
    conn_can = dblink_conn("CancionETL")
    conn_int = dblink_conn("Interacciones")

    # dim_ubicacion
    ejecutar(URI_OLAP, f"""
        INSERT INTO dim_ubicacion (continente, pais, estado)
        SELECT DISTINCT t.continente, t.pais, t.estado
        FROM dblink('{conn_usr}', 'SELECT continente, pais, estado FROM ubicacion')
        AS t(continente VARCHAR, pais VARCHAR, estado VARCHAR)
        ON CONFLICT (continente, pais, estado) DO NOTHING;
    """, "dim_ubicacion")

    # dim_dispositivo
    ejecutar(URI_OLAP, f"""
        INSERT INTO dim_dispositivo (tipo_dispositivo, sistema_operativo, idioma_dispositivo)
        SELECT DISTINCT t.tipo_dispositivo, t.sistema_operativo, t.idioma_dispositivo
        FROM dblink('{conn_int}', 'SELECT tipo_dispositivo, sistema_operativo, idioma_dispositivo FROM dispositivo')
        AS t(tipo_dispositivo VARCHAR, sistema_operativo VARCHAR, idioma_dispositivo VARCHAR)
        ON CONFLICT (tipo_dispositivo, sistema_operativo, idioma_dispositivo) DO NOTHING;
    """, "dim_dispositivo")

    # dim_usuario
    ejecutar(URI_OLAP, f"""
        INSERT INTO dim_usuario (id_usuario, nombre, edad, rango_edad, tipo_membresia)
        SELECT DISTINCT id_usuario, nombre, edad, rango_edad, tipo_membresia
        FROM dblink('{conn_usr}', 'SELECT id_usuario, nombre, edad, rango_edad, tipo_membresia FROM usuario')
        AS t(id_usuario VARCHAR, nombre VARCHAR, edad INT, rango_edad VARCHAR, tipo_membresia VARCHAR)
        ON CONFLICT (id_usuario) DO NOTHING;
    """, "dim_usuario")

    # dim_cancion
    ejecutar(URI_OLAP, f"""
        INSERT INTO dim_cancion (id_autor, duracion, idioma, genero, tema)
        SELECT DISTINCT t.id_autor, t.duracion, t.idioma, t.genero, t.tema
        FROM dblink('{conn_can}', 'SELECT id_autor, duracion, idioma, genero, tema FROM cancion')
        AS t(id_autor VARCHAR, duracion INT, idioma VARCHAR, genero VARCHAR, tema VARCHAR)
        ON CONFLICT (id_autor, duracion, idioma, genero, tema) DO NOTHING;
    """, "dim_cancion")

    # dim_fecha_hora (Histórico)
    ejecutar(URI_OLAP, f"""
        INSERT INTO dim_fecha_hora (fecha, dia, mes, trimestre, anio, dia_semana)
        SELECT DISTINCT
            fecha, dia, mes,
            EXTRACT(QUARTER FROM fecha)::INT AS trimestre,
            anio,
            CASE EXTRACT(DOW FROM fecha)
                WHEN 0 THEN 'Domingo' WHEN 1 THEN 'Lunes' WHEN 2 THEN 'Martes'
                WHEN 3 THEN 'Miércoles' WHEN 4 THEN 'Jueves' WHEN 5 THEN 'Viernes'
                WHEN 6 THEN 'Sábado'
            END AS dia_semana
        FROM dblink('{conn_int}', 'SELECT fecha, dia, mes, anio FROM fecha')
        AS t(fecha DATE, dia INT, mes INT, anio INT)
        ON CONFLICT (fecha) DO NOTHING;
    """, "dim_fecha_hora (histórico)")

    extender_fechas_futuras(meses=6)


def extender_fechas_futuras(meses=6):
    sql = f"""
        INSERT INTO dim_fecha_hora (fecha, dia, mes, trimestre, anio, dia_semana)
        SELECT
            d::DATE,
            EXTRACT(DAY   FROM d)::INT,
            EXTRACT(MONTH FROM d)::INT,
            EXTRACT(QUARTER FROM d)::INT,
            EXTRACT(YEAR  FROM d)::INT,
            CASE EXTRACT(DOW FROM d)
                WHEN 0 THEN 'Domingo' WHEN 1 THEN 'Lunes' WHEN 2 THEN 'Martes'
                WHEN 3 THEN 'Miércoles' WHEN 4 THEN 'Jueves' WHEN 5 THEN 'Viernes'
                WHEN 6 THEN 'Sábado'
            END
        FROM generate_series(CURRENT_DATE, CURRENT_DATE + INTERVAL '{meses} months', INTERVAL '1 day') AS d
        ON CONFLICT (fecha) DO NOTHING;
    """
    ejecutar(URI_OLAP, sql, f"dim_fecha_hora (+{meses} meses de proyección)")


def cargar_usuario_sistema_dss():
    sql = """
        INSERT INTO dim_usuario (id_usuario, nombre, edad, rango_edad, tipo_membresia)
        VALUES ('SYSTEM_DSS', 'Motor de Simulación DSS', 0, 'N/A', 'sistema')
        ON CONFLICT (id_usuario) DO NOTHING;
    """
    ejecutar(URI_OLAP, sql, "usuario sintético SYSTEM_DSS")


def cargar_dispositivo_sistema_dss():
    sql = """
        INSERT INTO dim_dispositivo (tipo_dispositivo, sistema_operativo, idioma_dispositivo)
        VALUES ('Agregado', 'N/A', 'N/A')
        ON CONFLICT (tipo_dispositivo, sistema_operativo, idioma_dispositivo) DO NOTHING;
    """
    ejecutar(URI_OLAP, sql, "dispositivo sintético (Agregado)")


# ─────────────────────────────────────────────────────────────────
# Manejo de índices secundarios de hechos_*
# ─────────────────────────────────────────────────────────────────
INDICES_HECHOS = {
    "hechos_interacciones": [           # ← CORREGIDO (antes: "hechos_interinteractions")
        ("idx_hechos_inter_usuario", "id_usuario"),
        ("idx_hechos_inter_cancion", "id_cancion"),
        ("idx_hechos_inter_fecha",   "id_fecha_hora"),
    ],
    "hechos_adquisicion": [
        ("idx_hechos_adquis_usuario",   "id_usuario"),
        ("idx_hechos_adquis_ubicacion", "id_ubicacion"),
    ],
}


def quitar_indices_hechos():
    print("\n    Quitando índices secundarios de hechos_* para acelerar la carga...")
    for _, indices in INDICES_HECHOS.items():
        for nombre, _col in indices:
            ejecutar(URI_OLAP, f"DROP INDEX IF EXISTS {nombre};", f"drop {nombre}")


def recrear_indices_hechos():
    print("\n    Recreando índices secundarios de hechos_*...")
    for tabla, indices in INDICES_HECHOS.items():
        for nombre, columna in indices:
            ejecutar(URI_OLAP, f"CREATE INDEX IF NOT EXISTS {nombre} ON {tabla}({columna});", f"create {nombre}")


# ─────────────────────────────────────────────────────────────────
# PASO 4: hechos_interacciones
# ─────────────────────────────────────────────────────────────────
def cargar_hechos_interacciones():
    print("\n[4] Cargando hechos_interacciones (JOIN nativo incremental)...")
    t = time.time()

    conn_usr = dblink_conn("UsersETL")
    conn_int = dblink_conn("Interacciones")
    conn_can = dblink_conn("CancionETL")

    sql_temp = f"""
    DO $$
    BEGIN
    DROP TABLE IF EXISTS tmp_usuario;
    CREATE TEMP TABLE tmp_usuario AS
        SELECT du.id_usuario, ub_olap.id_ubicacion
        FROM dblink('{conn_usr}',
            'SELECT u.id_usuario, ub.continente, ub.pais, ub.estado
             FROM usuario u JOIN ubicacion ub ON u.id_ubicacion = ub.id_ubicacion')
        AS u(id_usuario VARCHAR, continente VARCHAR, pais VARCHAR, estado VARCHAR)
        JOIN dim_ubicacion ub_olap
            ON ub_olap.continente = u.continente
           AND ub_olap.pais       = u.pais
           AND ub_olap.estado     = u.estado
        JOIN dim_usuario du ON du.id_usuario = u.id_usuario;

    DROP TABLE IF EXISTS tmp_dispositivo;
    CREATE TEMP TABLE tmp_dispositivo AS
        SELECT src.id_dispositivo_src, dd.id_dispositivo
        FROM dblink('{conn_int}',
            'SELECT id_dispositivo, tipo_dispositivo, sistema_operativo, idioma_dispositivo FROM dispositivo')
        AS src(id_dispositivo_src BIGINT, tipo_dispositivo VARCHAR, sistema_operativo VARCHAR, idioma_dispositivo VARCHAR)
        JOIN dim_dispositivo dd
            ON dd.tipo_dispositivo   = src.tipo_dispositivo
           AND dd.sistema_operativo  = src.sistema_operativo
           AND dd.idioma_dispositivo = src.idioma_dispositivo;

    DROP TABLE IF EXISTS tmp_fecha;
    CREATE TEMP TABLE tmp_fecha AS
        SELECT src.id_fecha_src, dfh.id_fecha_hora
        FROM dblink('{conn_int}', 'SELECT id_fecha, fecha FROM fecha')
        AS src(id_fecha_src BIGINT, fecha DATE)
        JOIN dim_fecha_hora dfh ON dfh.fecha = src.fecha;

    DROP TABLE IF EXISTS tmp_cancion;
    CREATE TEMP TABLE tmp_cancion AS
        SELECT src.id_cancion_src, dc.id_cancion
        FROM dblink('{conn_can}',
            'SELECT id_cancion, id_autor, duracion, idioma, genero, tema FROM cancion')
        AS src(id_cancion_src INT, id_autor VARCHAR, duracion INT, idioma VARCHAR, genero VARCHAR, tema VARCHAR)
        JOIN dim_cancion dc
            ON dc.id_autor = src.id_autor
           AND dc.duracion  = src.duracion
           AND dc.idioma    = src.idioma
           AND dc.genero    = src.genero
           AND dc.tema      = src.tema;

    CREATE INDEX ON tmp_usuario(id_usuario);
    CREATE INDEX ON tmp_dispositivo(id_dispositivo_src);
    CREATE INDEX ON tmp_fecha(id_fecha_src);
    CREATE INDEX ON tmp_cancion(id_cancion_src);

    END $$;
    """

    conn = psycopg2.connect(URI_OLAP)
    conn.autocommit = True
    cur = conn.cursor()

    print("   Creando tablas temporales de lookup...")
    cur.execute(sql_temp)
    print(f"   Tablas temporales listas en {time.time()-t:.1f}s")

    id_fecha_log = ejecutar_fetch(URI_OLAP, "SELECT id_fecha_hora FROM dim_fecha_hora WHERE fecha = CURRENT_DATE;")[0][0]

    LOTE = 1_000_000
    last_id = 0

    conn_check = psycopg2.connect(URI_OLAP)
    try:
        with conn_check.cursor() as c_id:
            c_id.execute("SELECT COALESCE(COUNT(*), 0) FROM hechos_interacciones;")
            total_ya_existente = c_id.fetchone()[0]
            if total_ya_existente >= 1_000_000:
                last_id = 1_000_000  # Salta el primer millón histórico
    finally:
        conn_check.close()

    total_ok = 0
    total_err = 0
    lote_n = 1

    print(f"   Insertando hechos en lotes. Iniciando desde id_interaccion > {last_id}...")
    while True:
        t_lote = time.time()
        sql = f"""
        WITH batch AS (
            SELECT * FROM dblink('{conn_int}',
                'SELECT id_interaccion, id_fecha, id_dispositivo, id_usuario, id_cancion,
                        tiempo_reproduccion, dio_like, dio_dislike, descargada
                 FROM interacciones
                 WHERE id_interaccion > {last_id}
                 ORDER BY id_interaccion
                 LIMIT {LOTE}')
            AS i(id_interaccion BIGINT, id_fecha BIGINT, id_dispositivo BIGINT, id_usuario VARCHAR,
                 id_cancion VARCHAR, tiempo_reproduccion INT,
                 dio_like BOOLEAN, dio_dislike BOOLEAN, descargada BOOLEAN)
        ),
        matched AS (
            SELECT
                b.*,
                tf.id_fecha_hora,
                tu.id_usuario     AS du_id_usuario,
                tu.id_ubicacion   AS du_id_ubicacion,
                td.id_dispositivo AS dd_id_dispositivo,
                tc.id_cancion     AS dc_id_cancion
            FROM batch b
            LEFT JOIN tmp_fecha       tf ON tf.id_fecha_src       = b.id_fecha
            LEFT JOIN tmp_dispositivo td ON td.id_dispositivo_src = b.id_dispositivo
            LEFT JOIN tmp_usuario     tu ON tu.id_usuario         = b.id_usuario
            LEFT JOIN tmp_cancion     tc ON tc.id_cancion_src     = b.id_cancion::INT
        ),
        ins_ok AS (
            INSERT INTO hechos_interacciones
                (id_fecha_hora, id_usuario, id_cancion, id_ubicacion, id_dispositivo,
                 tiempo_reproduccion, dio_like, dio_dislike, descargada)
            SELECT id_fecha_hora, du_id_usuario, dc_id_cancion, du_id_ubicacion, dd_id_dispositivo,
                   tiempo_reproduccion, dio_like::INT, dio_dislike::INT, descargada::INT
            FROM matched
            WHERE id_fecha_hora IS NOT NULL AND du_id_usuario IS NOT NULL AND dc_id_cancion IS NOT NULL
              AND du_id_ubicacion IS NOT NULL AND dd_id_dispositivo IS NOT NULL
            RETURNING 1
        ),
        ins_err AS (
            INSERT INTO hechos_data_quality (id_fecha_hora, entidad_afectada, tipo_error, detalle_error, severidad)
            SELECT
                {id_fecha_log},
                'hechos_interacciones',
                CASE
                    WHEN id_fecha_hora     IS NULL THEN 'fecha_no_encontrada'
                    WHEN du_id_usuario     IS NULL THEN 'usuario_no_encontrado'
                    WHEN dc_id_cancion     IS NULL THEN 'cancion_no_encontrada'
                    WHEN du_id_ubicacion   IS NULL THEN 'ubicacion_no_encontrada'
                    WHEN dd_id_dispositivo IS NULL THEN 'dispositivo_no_encontrado'
                END,
                format('id_interaccion=%s id_fecha=%s id_usuario=%s id_cancion=%s id_dispositivo=%s',
                       id_interaccion, id_fecha, id_usuario, id_cancion, id_dispositivo),
                3
            FROM matched
            WHERE id_fecha_hora IS NULL OR du_id_usuario IS NULL OR dc_id_cancion IS NULL
               OR du_id_ubicacion IS NULL OR dd_id_dispositivo IS NULL
            RETURNING 1
        )
        SELECT (SELECT MAX(id_interaccion) FROM batch),
               (SELECT COUNT(*) FROM batch),
               (SELECT COUNT(*) FROM ins_ok),
               (SELECT COUNT(*) FROM ins_err);
        """
        cur.execute(sql)
        max_id, n_batch, n_ok, n_err = cur.fetchone()
        if n_batch == 0:
            break

        last_id = max_id
        total_ok += n_ok
        total_err += n_err
        print(f"   [Lote #{lote_n}] hasta id_interaccion={last_id:,} | "
              f"OK {n_ok:,} | DQ {n_err:,} | {time.time()-t_lote:.1f}s | acumulado {total_ok:,}")

        if n_batch < LOTE:
            break
        lote_n += 1

    cur.close()
    conn.close()
    print(f"   hechos_interacciones completo en {(time.time()-t)/60:.1f} min "
          f"({total_ok:,} insertadas, {total_err:,} con incidencias en hechos_data_quality)")


# ─────────────────────────────────────────────────────────────────
# PASO 5: hechos_adquisicion (Incremental)
# ─────────────────────────────────────────────────────────────────
def cargar_hechos_adquisicion():
    print("\n[5] Cargando hechos_adquisicion de forma incremental...")
    conn_usr = dblink_conn("UsersETL")

    ejecutar(URI_OLAP, f"""
        INSERT INTO hechos_adquisicion
            (id_fecha_hora, id_usuario, id_ubicacion, id_dispositivo, usuario_registrado)
        SELECT
            1,
            du.id_usuario,
            dub.id_ubicacion,
            1,
            1
        FROM dblink('{conn_usr}',
            'SELECT u.id_usuario, ub.continente, ub.pais, ub.estado
             FROM usuario u JOIN ubicacion ub ON u.id_ubicacion = ub.id_ubicacion')
        AS src(id_usuario VARCHAR, continente VARCHAR, pais VARCHAR, estado VARCHAR)
        JOIN dim_usuario   du  ON du.id_usuario   = src.id_usuario
        JOIN dim_ubicacion dub ON dub.continente  = src.continente
                               AND dub.pais        = src.pais
                               AND dub.estado      = src.estado
        WHERE NOT EXISTS (
            SELECT 1 FROM hechos_adquisicion ha WHERE ha.id_usuario = du.id_usuario
        );
    """, "hechos_adquisicion")


# ─────────────────────────────────────────────────────────────────
# PASO 6: hechos_simulaciones
# ─────────────────────────────────────────────────────────────────
def cargar_hechos_simulaciones(meses_proyeccion=3):
    print(f"\n[6] Re-calculando hechos_simulaciones ({meses_proyeccion} meses)...")
    ejecutar(URI_OLAP, "TRUNCATE TABLE hechos_simulaciones RESTART IDENTITY;", "Limpieza previa a simulación")

    sql = f"""
    INSERT INTO hechos_simulaciones
        (id_escenario, id_ubicacion, id_fecha_hora, id_usuario, id_dispositivo,
         kpi_proyectado_valor, metodo_proyeccion)
    WITH base AS (
        SELECT m.id_ubicacion, AVG(m.conteo_mensual) AS kpi_base
        FROM (
            SELECT hi.id_ubicacion, dfh.anio, dfh.mes, COUNT(*) AS conteo_mensual
            FROM hechos_interacciones hi
            JOIN dim_fecha_hora dfh ON dfh.id_fecha_hora = hi.id_fecha_hora
            WHERE dfh.fecha >= (CURRENT_DATE - INTERVAL '3 months')
            GROUP BY hi.id_ubicacion, dfh.anio, dfh.mes
        ) m
        GROUP BY m.id_ubicacion
    ),
    escenarios AS (
        SELECT id_escenario, nombre_escenario,
               CASE nombre_escenario
                   WHEN 'Optimista' THEN 1.15
                   WHEN 'Pesimista' THEN 0.90
                   ELSE 1.00
               END AS factor_mensual
        FROM dim_escenario
    ),
    futuro AS (
        SELECT id_fecha_hora, fecha
        FROM dim_fecha_hora
        WHERE fecha > CURRENT_DATE AND dia = 1
        ORDER BY fecha
        LIMIT {meses_proyeccion}
    )
    SELECT
        e.id_escenario,
        b.id_ubicacion,
        f.id_fecha_hora,
        (SELECT id_usuario FROM dim_usuario WHERE id_usuario = 'SYSTEM_DSS'),
        (SELECT id_dispositivo FROM dim_dispositivo
            WHERE tipo_dispositivo = 'Agregado' AND sistema_operativo = 'N/A' AND idioma_dispositivo = 'N/A'),
        ROUND(
            b.kpi_base * POWER(
                e.factor_mensual,
                (EXTRACT(YEAR FROM f.fecha) - EXTRACT(YEAR FROM CURRENT_DATE)) * 12
                + (EXTRACT(MONTH FROM f.fecha) - EXTRACT(MONTH FROM CURRENT_DATE))
            ),
            2),
        'tasa_crecimiento_simple'
    FROM base b
    CROSS JOIN escenarios e
    CROSS JOIN futuro f;
    """
    ejecutar(URI_OLAP, sql, "hechos_simulaciones")


# ─────────────────────────────────────────────────────────────────
# ORQUESTADOR PRINCIPAL
# ─────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    T0 = time.time()
    print("=" * 60)
    print(" ETL DSS — EJECUCIÓN 100% INCREMENTAL")
    print("=" * 60)

    instalar_dblink()
    cargar_dim_escenario()

    # limpiar_olap()  ← descomentar solo si quieres resetear todo desde cero

    cargar_dimensiones()
    cargar_usuario_sistema_dss()
    cargar_dispositivo_sistema_dss()

    quitar_indices_hechos()
    cargar_hechos_interacciones()
    cargar_hechos_adquisicion()
    recrear_indices_hechos()

    cargar_hechos_simulaciones(meses_proyeccion=3)

    print(f"\n{'='*60}")
    print(f" COMPLETADO CON ÉXITO EN {(time.time()-T0)/60:.1f} MINUTOS")
    print(f"{'='*60}")

 ETL DSS — EJECUCIÓN 100% INCREMENTAL

[0] Instalando extensión dblink en OLAP_Operations...
   [OK] dblink instalado → ? filas en 0.8s

[2] Cargando dim_escenario (catálogo de escenarios DSS)...
   [OK] dim_escenario → ? filas en 0.7s

[3] Cargando dimensiones via dblink de forma incremental...
   [OK] dim_ubicacion → ? filas en 0.8s
   [OK] dim_dispositivo → ? filas en 0.7s
   [OK] dim_usuario → ? filas en 2.0s
   [OK] dim_cancion → ? filas en 1.6s
   [OK] dim_fecha_hora (histórico) → ? filas en 0.7s
   [OK] dim_fecha_hora (+6 meses de proyección) → ? filas en 0.7s
   [OK] usuario sintético SYSTEM_DSS → ? filas en 0.7s
   [OK] dispositivo sintético (Agregado) → ? filas en 0.6s

    Quitando índices secundarios de hechos_* para acelerar la carga...
   [OK] drop idx_hechos_inter_usuario → ? filas en 0.7s
   [OK] drop idx_hechos_inter_cancion → ? filas en 0.6s
   [OK] drop idx_hechos_inter_fecha → ? filas en 0.7s
   [OK] drop idx_hechos_adquis_usuario → ? filas en 0.6s
   [OK] drop idx_